## 05 - XDec Pilot Timing (k=3,4)

### Aim
* Before committing to the full XDec Stage-0 stability sweep (default `k=3..12`) on the full 182-sample TCGA-PAAD cohort, time a small pilot restricted to `--k-range 3,4` on this actual laptop (16GB, M3), and use that measurement to decide whether the full sweep needs to be partitioned (per notebook 04's resource-management plan) rather than run as one blocking call.

### Input
* `data/processed/TCGA-PAAD_counts_raw.tsv` (raw counts, notebook 02) - the real bulk matrix XDec is designed to consume directly.
* `tools/XDec_SM/python/xdec.py` - the cloned `BRL-BCM/XDec_SM` repo (verified Python; see notebook 04).
* A **placeholder** 283-gene probe list built here by selecting the top-variance genes from the counts matrix - NOT the published Stage-0 gene set (that choice is still an open decision in notebook 04). This pilot is about timing the code path, not producing biologically meaningful results.

### Output
* `data/deconvolution/xdec_pilot/` - Stage-0 outputs (`stability_scores.tsv`, `optimal_k.txt`, etc.) for `k=3,4` only, plus the placeholder probe list used to generate them.
* A timed elapsed-seconds measurement for the pilot, and a projected elapsed time for the full default `k=3..12` sweep, extrapolated from the exact inner-loop structure of `estimate_stability()` (read directly from the cloned source, not assumed).

### Steps
1. Idempotent setup: confirm the XDec_SM repo is cloned, and confirm/install its one missing dependency (`quadprog`) into the global `pixi` `python` environment - skipped if already present.
2. Build the placeholder (top-variance) 283-gene probe list, purely to give Stage 0 a realistically-sized gene set to run its stability loop on.
3. Run `xdec.py stage0 --k-range 3,4` against the real counts matrix, timed with `Sys.time()`.
4. Extrapolate to the full default sweep (`k=3..12`) using the confirmed inner-loop count (`num_subsets=3 x reps_per_subset=3` Stage-1 calls per k - read from `estimate_stability()`'s source, not guessed), and recommend whether to partition the full run by k.

### Result
* Not yet run - elapsed-time cells below will populate once this notebook is executed.

### Conclusion
* Not yet run. Once executed, this notebook's final cell states whether the full `k=3..12` sweep is safe to run as a single blocking call (projected <30 min) or should be partitioned by k into separate subprocess calls (per notebook 04's partitioning recommendation).

### Set working directory to the project root

In [1]:
# Jupyter's R kernel starts with the notebook's own folder as the working
# directory. This pipeline's root is progress/ - every path below is
# relative to that (same pattern as notebooks 00-04).
if (basename(getwd()) == "rnaseq_download") setwd(file.path("..", ".."))
getwd()

[1] "/Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer/progress"

### Setup: confirm XDec_SM repo + `quadprog` dependency

`quadprog` is technically optional - `xdec.py` falls back to a slower `scipy` SLSQP solver if it's missing - but the fallback is not representative of the tool's intended (and recommended) performance, so it's installed here for an accurate pilot measurement. This is the one step in this notebook that modifies the **global** `pixi` `python` environment (shared across projects, not local to this repo) - see notebook 04/00 for why this is the verified-safe install path (`pixi global install`, not `pip`/`remotes::install_github`).

In [2]:
xdec_repo <- "tools/XDec_SM"
if (!file.exists(file.path(xdec_repo, "python", "xdec.py"))) {
  dir.create(dirname(xdec_repo), recursive = TRUE, showWarnings = FALSE)
  system2("git", c("clone", "--depth", "1",
                    "https://github.com/BRL-BCM/XDec_SM.git", xdec_repo))
}
stopifnot(file.exists(file.path(xdec_repo, "python", "xdec.py")))

py_bin <- path.expand("~/.pixi/envs/python/bin/python")
stopifnot(file.exists(py_bin))

# system2() does not shell-quote multi-word elements of `args` - "import quadprog"
# as one element gets split by the shell into two words unless explicitly quoted.
quadprog_status <- system2(py_bin, c("-c", shQuote("import quadprog")),
                            stdout = FALSE, stderr = FALSE)
if (quadprog_status != 0) {
  message("quadprog not found in the pixi 'python' env - installing ",
          "(this modifies the GLOBAL pixi environment, shared across projects).")
  system2("pixi", c("global", "install", "quadprog", "--environment", "python"))
  quadprog_status <- system2(py_bin, c("-c", shQuote("import quadprog")),
                              stdout = FALSE, stderr = FALSE)
  stopifnot(quadprog_status == 0)
} else {
  message("quadprog already installed - skipping.")
}

quadprog already installed - skipping.



### Placeholder probe list (283 top-variance genes) - for timing only

Not the real Stage-0 informative gene set (that choice - published 283-gene list vs. re-derived from scRNA-seq - is still an open decision in notebook 04). Using `--probes` here just skips the t-test selection step so this pilot measures the stability loop in isolation, on a realistically-sized gene set (283 genes, matching the published set's size).

In [3]:
counts_path <- "data/processed/TCGA-PAAD_counts_raw.tsv"
counts <- read.delim(counts_path, check.names = FALSE, row.names = 1)

gene_var <- apply(counts, 1, var)
placeholder_genes <- names(sort(gene_var, decreasing = TRUE))[1:283]

pilot_dir <- "data/deconvolution/xdec_pilot"
dir.create(pilot_dir, recursive = TRUE, showWarnings = FALSE)

probes_path <- file.path(pilot_dir, "placeholder_probes_283.txt")
writeLines(placeholder_genes, probes_path)
length(placeholder_genes)

[1] 283

### Run the timed pilot: Stage 0, `--k-range 3,4`

In [4]:
xdec_script <- file.path(xdec_repo, "python", "xdec.py")

pilot_args <- c(
  xdec_script, "stage0",
  "--bulk", counts_path,
  "--probes", probes_path,
  "--k-range", "3,4",
  "--outdir", pilot_dir
)

t0 <- Sys.time()
pilot_log <- system2(py_bin, pilot_args, stdout = TRUE, stderr = TRUE)
t1 <- Sys.time()

elapsed_pilot_sec <- as.numeric(difftime(t1, t0, units = "secs"))
cat(pilot_log, sep = "\n")
cat(sprintf("\nPilot (k=3,4) elapsed: %.1f sec (%.1f min)\n",
            elapsed_pilot_sec, elapsed_pilot_sec / 60))

12:46:43  INFO      Loaded data/processed/TCGA-PAAD_counts_raw.tsv: 54603 genes × 182 samples
12:46:46  INFO      After normalisation: 54603 expressed genes × 182 bulk samples
12:46:46  INFO      Loaded 283 probes from data/deconvolution/xdec_pilot/placeholder_probes_283.txt
12:46:46  INFO      Saved 283 probes → data/deconvolution/xdec_pilot/selected_probes.txt
12:46:46  INFO      Stability: testing k=3 …
12:59:08  INFO        k= 3  stability=1.0000
12:59:08  INFO      Stability: testing k=4 …
13:57:13  INFO        k= 4  stability=1.0000
13:57:13  INFO      Saved  data/deconvolution/xdec_pilot/stability_scores.tsv  (2 × 1)
13:57:13  INFO      Recommended k = 3  (stability=1.0000)



Pilot (k=3,4) elapsed: 4258.1 sec (71.0 min)


### Extrapolate to the full default sweep (`k=3..12`) and decide on partitioning

`estimate_stability()` (read directly from `tools/XDec_SM/python/xdec.py`) runs `num_subsets=3 x reps_per_subset=3` = 9 inner Stage-1 calls per k, independent of which k values are tested. The pilot's `k-range 3,4` is 2 k values = 18 inner calls; the full default `k-range 3,4,...,12` is 10 k values = 90 inner calls - a confirmed 5x scale-up, not a guess.

In [5]:
pilot_inner_calls <- 2 * 9   # k-range 3,4
full_inner_calls  <- 10 * 9  # default k-range 3..12
extrapolation_factor <- full_inner_calls / pilot_inner_calls

projected_full_sec <- elapsed_pilot_sec * extrapolation_factor
cat(sprintf(
  "Projected full stability sweep (k=3..12, default): %.1f sec (%.1f min)\n",
  projected_full_sec, projected_full_sec / 60
))

if (projected_full_sec > 30 * 60) {
  cat("Over 30 min projected - partition the full sweep by k: loop over\n",
      "individual --k-range values as separate subprocess calls, writing\n",
      "each k's stability score to disk as it completes, instead of one\n",
      "blocking call for the whole range (see notebook 04's resource plan).\n")
} else {
  cat("Under 30 min projected - safe to run the full k-range in one call.\n")
}

Projected full stability sweep (k=3..12, default): 21290.5 sec (354.8 min)


Over 30 min projected - partition the full sweep by k: loop over
 individual --k-range values as separate subprocess calls, writing
 each k's stability score to disk as it completes, instead of one
 blocking call for the whole range (see notebook 04's resource plan).
